# Taller E1-02 — Perfilamiento de clientes con estadística descriptiva

**Estadística I — Ingeniería Industrial, UIS**

## Instrucciones generales

- Trabajas con una muestra propia, seleccionada mediante **Muestreo Aleatorio Simple (MAS)** a partir de una base de datos de clientes de una cadena de retail. Tu código estudiantil determina cuáles registros te tocaron — tus resultados serán distintos a los de tus compañeros, **eso es intencional**.
- Cada sección plantea una situación de negocio real, te da el **nombre de la función recomendada** y un enlace a su documentación — la llamada exacta la construyes tú.
- Dentro de cada sección, primero resuelves el **cálculo** (código) y después respondes una o más **preguntas de opción múltiple** que validan si interpretaste correctamente ese cálculo. Las preguntas están numeradas como *Pregunta X.Y*.
- La opción múltiple se responde como texto, ej. `respuesta_mcq1 <- "B"`.
- Al final hay una pregunta abierta de reflexión (revisada manualmente, no autocalificada).
- Antes de correr las celdas de verificación (`ottr::check`), sube **`anexos.zip`** al panel de Archivos de tu sesión de Colab (ver celda de configuración) — trae la carpeta de tests y la base de datos.


In [ ]:
# Si subiste anexos.zip (y no las carpetas/archivos ya descomprimidos), descomenta y corre esta linea:
# unzip("anexos.zip")

if (!dir.exists("tests") || !file.exists("clientes_tienda.csv")) {
  cat("AVISO: falta la carpeta 'tests' o 'clientes_tienda.csv'. Sube anexos.zip desde el panel de Archivos y descomprimelo antes de continuar.\n")
}

# Instala ottr (el paquete que corre las auto-verificaciones) solo si hace falta
if (!requireNamespace("ottr", quietly = TRUE)) install.packages("ottr")

In [ ]:
# ============================================================
# CONFIGURACIÓN INICIAL — NO MODIFIQUES ESTA CELDA (salvo la línea indicada)
# ============================================================

# --> Reemplaza el valor de abajo por tu código estudiantil (solo números)
codigo_estudiante <- 0000000

reseed <- function(offset) {
  set.seed(codigo_estudiante + offset)
  RNGkind("Mersenne-Twister", "Inversion", "Rejection")
}

datos_completos <- read.csv("clientes_tienda.csv", stringsAsFactors = FALSE)

# Selección de tu muestra mediante Muestreo Aleatorio Simple (MAS)
reseed(1)
tamano_muestra <- 120
indices_muestra <- sample(seq_len(nrow(datos_completos)), size = tamano_muestra)
muestra <- datos_completos[indices_muestra, ]

library(moments)

cat("Tu muestra tiene", nrow(muestra), "clientes.\n")
head(muestra)


Diccionario de datos de `muestra`:

| Variable | Tipo | Descripción |
|---|---|---|
| `region` | Categórica | Zona geográfica de residencia del cliente (Norte, Sur, Centro, Oriente), usada para decisiones de cobertura comercial. |
| `genero` | Categórica | Género con el que el cliente se identifica en su perfil de la tienda (Femenino, Masculino). |
| `tipo_membresia` | Categórica | Nivel del programa de fidelización del cliente (Bronce, Plata, Oro), asociado a distintos beneficios y descuentos. |
| `edad` | Numérica | Edad del cliente en años cumplidos, registrada al momento de la afiliación. |
| `antiguedad_meses` | Numérica | Meses transcurridos desde que el cliente se afilió al programa de fidelización. |
| `frecuencia_compra_mensual` | Numérica | Número de compras que el cliente realiza en un mes típico. |
| `gasto_mensual` | Numérica | Monto total gastado por el cliente en un mes típico (pesos colombianos). |


---
## 1. Regiones significativas para la expansión de campañas

La gerencia comercial está evaluando en qué ciudades abrir nuevos puntos de venta y dónde invertir el presupuesto de publicidad del próximo trimestre. Antes de tomar esa decisión, necesitan un diagnóstico objetivo de dónde está concentrada realmente la base de clientes actual — no basta con la percepción de los vendedores de cada zona, se necesita el dato.

**Función recomendada:** `table()` — [documentación](https://www.rdocumentation.org/packages/base/versions/3.6.2/topics/table). Para la frecuencia relativa, combínala con `prop.table()`.


**Pregunta 1.1 (código):** Construye la tabla de frecuencias de `region` en tu muestra en una variable llamada `tabla_region`. Guarda en `region_mas_frecuente` el nombre de la región con mayor frecuencia, y en `frecuencia_relativa_moda` su frecuencia relativa (proporción, no porcentaje).

In [ ]:
# BEGIN SOLUTION
tabla_region <- table(muestra$region)
region_mas_frecuente <- names(which.max(tabla_region))
frecuencia_relativa_moda <- as.numeric(max(prop.table(tabla_region)))
# END SOLUTION
tabla_region

In [ ]:
. = ottr::check("tests/p2_frecuencias.R")

**Pregunta 1.2 (opción múltiple):** ¿qué tan concentrada está la base de clientes en su región más frecuente?

- A) Muy concentrada: esa región supera el 50% de los clientes.
- B) Moderadamente concentrada: esa región está entre 25% y 50% de los clientes.
- C) Poco concentrada: esa región está por debajo del 25% de los clientes.
- D) Los clientes están distribuidos exactamente igual entre todas las regiones.

Guarda la letra en `respuesta_mcq1`.


In [ ]:
# BEGIN SOLUTION
respuesta_mcq1 <- {f <- max(prop.table(table(muestra$region))); if (f > 0.5) "A" else if (f >= 0.25) "B" else "C"}  # en la mayoría de las muestras: "B"
# END SOLUTION
respuesta_mcq1

In [ ]:
. = ottr::check("tests/p2_mcq1_concentracion.R")

---
## 2. Comparación de variabilidad entre edad y gasto

El área comercial está diseñando una estrategia de precios y quiere saber en cuál variable los clientes son más heterogéneos: en edad (para decidir si una sola campaña le sirve a todos los rangos etarios) o en gasto (para decidir si conviene una política de precios única o diferenciada por cliente). Como estas variables están en escalas distintas (años vs. pesos), no basta con comparar sus desviaciones estándar directamente — se necesita una medida de dispersión relativa.

**Funciones recomendadas:** `mean()` y `sd()` — [documentación de mean](https://www.rdocumentation.org/packages/base/versions/3.6.2/topics/mean), [documentación de sd](https://www.rdocumentation.org/packages/stats/versions/3.6.2/topics/sd). El coeficiente de variación se calcula como `sd / mean`.


**Pregunta 2.1 (código):** Calcula media, desviación estándar y coeficiente de variación para `edad` y para `gasto_mensual` en tu muestra: `media_edad`, `sd_edad`, `cv_edad`, `media_gasto`, `sd_gasto`, `cv_gasto`.

In [ ]:
# BEGIN SOLUTION
media_edad <- mean(muestra$edad)
sd_edad <- sd(muestra$edad)
cv_edad <- sd_edad / media_edad
media_gasto <- mean(muestra$gasto_mensual)
sd_gasto <- sd(muestra$gasto_mensual)
cv_gasto <- sd_gasto / media_gasto
# END SOLUTION
c(cv_edad = cv_edad, cv_gasto = cv_gasto)

In [ ]:
. = ottr::check("tests/p2_media_sd_cv.R")

**Pregunta 2.2 (opción múltiple):** ¿cuál de las dos variables tiene mayor dispersión relativa?

- A) `edad`, porque su desviación estándar en años es más fácil de interpretar.
- B) `gasto_mensual`, porque su coeficiente de variación es notablemente mayor.
- C) Ambas tienen dispersión relativa prácticamente igual.
- D) No se puede comparar la dispersión de variables en escalas distintas.

Guarda la letra en `respuesta_mcq2`.


In [ ]:
# BEGIN SOLUTION
respuesta_mcq2 <- {ce <- sd(muestra$edad)/mean(muestra$edad); cg <- sd(muestra$gasto_mensual)/mean(muestra$gasto_mensual); if (cg > 1.2 * ce) "B" else if (ce > 1.2 * cg) "A" else "C"}  # en la mayoría de las muestras: "B"
# END SOLUTION
respuesta_mcq2

In [ ]:
. = ottr::check("tests/p2_mcq2_dispersion.R")

**Pregunta 2.3 (opción múltiple):** si el gasto tiene una dispersión relativa mucho mayor que la edad, ¿qué implica esto para diseñar una política de precios única para todos los clientes?

- A) Que una política de precios única probablemente ignora diferencias importantes entre clientes.
- B) Que todos los clientes gastan de forma muy similar y una política única es adecuada.
- C) Que la edad es la variable más relevante para segmentar precios.
- D) Que el coeficiente de variación no aplica a variables de gasto monetario.

Guarda la letra en `respuesta_mcq3`.


In [ ]:
# BEGIN SOLUTION
respuesta_mcq3 <- "A"
# END SOLUTION

In [ ]:
. = ottr::check("tests/p2_mcq3_implicacion_cv.R")

---
## 3. Forma de la distribución del gasto: ¿hay clientes que la distorsionan?

El equipo financiero quiere fijar una meta de gasto mensual "típico" por cliente para sus proyecciones de ingresos. Antes de usar el promedio como esa meta, necesitan saber si la distribución del gasto es simétrica (donde la media representa bien al cliente típico) o si hay unos pocos clientes de gasto muy alto que están inflando el promedio y distorsionando esa referencia.

**Funciones recomendadas:** `skewness()` y `kurtosis()` del paquete `moments` (ya cargado) — [documentación](https://www.rdocumentation.org/packages/moments/versions/0.14.1/topics/skewness).


**Pregunta 3.1 (código):** Calcula la asimetría y la curtosis de `gasto_mensual` en tu muestra: `asimetria_gasto`, `curtosis_gasto`.

In [ ]:
# BEGIN SOLUTION
asimetria_gasto <- skewness(muestra$gasto_mensual)
curtosis_gasto <- kurtosis(muestra$gasto_mensual)
# END SOLUTION
c(asimetria_gasto, curtosis_gasto)

In [ ]:
. = ottr::check("tests/p2_asimetria_curtosis.R")

**Pregunta 3.2 (opción múltiple):** ¿qué te dice el valor de la asimetría sobre la meta de gasto "típico" que debería usar financiero?

- A) La distribución es simétrica; el promedio representa bien al cliente típico.
- B) Es asimétrica a la izquierda: hay clientes con gastos inusualmente bajos que bajan el promedio.
- C) Es asimétrica a la derecha: hay clientes con gastos inusualmente altos que inflan el promedio.
- D) La asimetría no se puede interpretar sin construir antes un histograma.

Guarda la letra en `respuesta_mcq4`.


In [ ]:
# BEGIN SOLUTION
respuesta_mcq4 <- {s <- moments::skewness(muestra$gasto_mensual); if (s > 0.5) "C" else if (s < -0.5) "B" else "A"}  # en la mayoría de las muestras: "C"
# END SOLUTION
respuesta_mcq4

In [ ]:
. = ottr::check("tests/p2_mcq4_asimetria.R")

**Pregunta 3.3 (opción múltiple):** ¿qué indica un valor de curtosis alto (leptocúrtico) como el que calculaste?

- A) Un pico marcado y colas pesadas: hay más clientes con gastos extremos de lo que esperaría una distribución normal.
- B) Que la distribución es perfectamente uniforme.
- C) Que la curtosis baja siempre acompaña a una asimetría negativa.
- D) Que la curtosis mide exactamente lo mismo que la asimetría, solo en otra escala.

Guarda la letra en `respuesta_mcq5`.


In [ ]:
# BEGIN SOLUTION
respuesta_mcq5 <- "A"
# END SOLUTION

In [ ]:
. = ottr::check("tests/p2_mcq5_curtosis.R")

---
## 4. Detección de clientes con antigüedad atípica

El área de fidelización quiere premiar a los clientes más antiguos con beneficios especiales ("embajadores de marca"), pero necesita un criterio objetivo para decidir a partir de qué antigüedad un cliente se considera atípicamente leal, en vez de escoger una cifra arbitraria a ojo.

**Función recomendada:** `quantile()` y/o `summary()` — [documentación de quantile](https://www.rdocumentation.org/packages/stats/versions/3.6.2/topics/quantile).


**Pregunta 4.1 (código):** Calcula, para `antiguedad_meses`: el primer cuartil (`q1_antiguedad`), la mediana (`mediana_antiguedad`), el tercer cuartil (`q3_antiguedad`) y el rango intercuartílico (`iqr_antiguedad`).

In [ ]:
# BEGIN SOLUTION
q1_antiguedad <- as.numeric(quantile(muestra$antiguedad_meses, 0.25))
mediana_antiguedad <- median(muestra$antiguedad_meses)
q3_antiguedad <- as.numeric(quantile(muestra$antiguedad_meses, 0.75))
iqr_antiguedad <- q3_antiguedad - q1_antiguedad
# END SOLUTION
c(q1_antiguedad, mediana_antiguedad, q3_antiguedad, iqr_antiguedad)

In [ ]:
. = ottr::check("tests/p2_cinco_numeros.R")

**Pregunta 4.2 (opción múltiple):** usando la regla de Tukey (un dato es atípico si supera `Q3 + 1.5*IQR`), ¿cuál es el criterio correcto para el umbral de antigüedad atípica?

- A) Aproximadamente el doble de la mediana.
- B) `Q3 + 1.5 * IQR`: el valor a partir del cual la regla de Tukey considera atípico un dato por arriba.
- C) Exactamente `Q3`.
- D) La media más una desviación estándar.

Guarda la letra en `respuesta_mcq6`.


In [ ]:
# BEGIN SOLUTION
respuesta_mcq6 <- "B"
# END SOLUTION

In [ ]:
. = ottr::check("tests/p2_mcq6_outliers.R")

**Pregunta 4.3 (código):** Calcula el valor mínimo (`minimo_antiguedad`) y máximo (`maximo_antiguedad`) de `antiguedad_meses` en tu muestra. Usando la regla de Tukey y los cuartiles que ya calculaste, calcula también el límite inferior (`limite_inferior_antiguedad` = `Q1 - 1.5*IQR`) y el límite superior (`limite_superior_antiguedad` = `Q3 + 1.5*IQR`) para valores atípicos.

In [ ]:
# BEGIN SOLUTION
minimo_antiguedad <- min(muestra$antiguedad_meses)
maximo_antiguedad <- max(muestra$antiguedad_meses)
limite_inferior_antiguedad <- q1_antiguedad - 1.5 * iqr_antiguedad
limite_superior_antiguedad <- q3_antiguedad + 1.5 * iqr_antiguedad
# END SOLUTION
c(minimo_antiguedad, limite_inferior_antiguedad, limite_superior_antiguedad, maximo_antiguedad)

In [ ]:
. = ottr::check("tests/p2_limites_atipicos.R")

**Pregunta 4.4 (opción múltiple):** comparando el mínimo con el límite inferior, y el máximo con el límite superior, ¿hay valores atípicos de antigüedad en tu muestra?

- A) No hay atípicos ni inferiores ni superiores: todos los valores están dentro de los límites.
- B) Hay atípicos inferiores (el mínimo está por debajo del límite inferior), pero no superiores.
- C) Hay atípicos superiores (el máximo está por encima del límite superior), pero no inferiores.
- D) Hay atípicos tanto inferiores como superiores.

Guarda la letra en `respuesta_mcq7`.


In [ ]:
# BEGIN SOLUTION
respuesta_mcq7 <- {x <- muestra$antiguedad_meses; q <- quantile(x, c(.25, .75)); r <- IQR(x); inf <- min(x) < q[1] - 1.5*r; sup <- max(x) > q[2] + 1.5*r; if (inf && sup) "D" else if (sup) "C" else if (inf) "B" else "A"}  # en la mayoría de las muestras: "C"
# END SOLUTION
respuesta_mcq7

In [ ]:
. = ottr::check("tests/p2_mcq7_atipicos.R")

---
## 5. ¿La membresía se distribuye distinto entre géneros?

El equipo de fidelización está diseñando una campaña para incentivar el ascenso de nivel de membresía y quiere saber si debe diseñar mensajes diferenciados por género, o si el nivel de membresía se distribuye de forma similar entre hombres y mujeres y un solo mensaje sirve para todos.

**Función recomendada:** `table()` con dos variables, que genera automáticamente una tabla de contingencia — [documentación](https://www.rdocumentation.org/packages/base/versions/3.6.2/topics/table).


**Pregunta 5.1 (código):** Construye la tabla de contingencia entre `genero` y `tipo_membresia` en la variable `tabla_contingencia`. Guarda en `prop_oro_femenino` la proporción de clientes **Femenino** que tienen membresía **Oro** (del total de clientes Femenino en tu muestra, qué fracción es Oro).

In [ ]:
# BEGIN SOLUTION
tabla_contingencia <- table(muestra$genero, muestra$tipo_membresia)
prop_oro_femenino <- as.numeric(tabla_contingencia["Femenino", "Oro"] / sum(tabla_contingencia["Femenino", ]))
# END SOLUTION
tabla_contingencia

In [ ]:
. = ottr::check("tests/p2_contingencia.R")

**Pregunta 5.2 (opción múltiple):** ¿qué tipo de análisis permite una tabla de contingencia que una tabla de frecuencias de una sola variable no permite?

- A) Calcular el promedio de una variable numérica.
- B) Ver cómo se distribuye una variable categórica dentro de cada categoría de otra variable.
- C) Detectar valores atípicos.
- D) Calcular la asimetría de una distribución.

Guarda la letra en `respuesta_mcq8`.


In [ ]:
# BEGIN SOLUTION
respuesta_mcq8 <- "B"
# END SOLUTION

In [ ]:
. = ottr::check("tests/p2_mcq8_contingencia_concepto.R")

**Pregunta 5.3 (opción múltiple):** independientemente del resultado exacto en tu muestra — si las proporciones de membresía Oro fueran prácticamente iguales entre Femenino y Masculino, ¿qué sugeriría esto sobre la relación entre las dos variables?

- A) Que existe una fuerte relación entre género y nivel de membresía.
- B) Que el género no parece influir en el nivel de membresía alcanzado.
- C) Que todos los clientes Oro son necesariamente del mismo género.
- D) Que la tabla de contingencia no sirve para este tipo de pregunta.

Guarda la letra en `respuesta_mcq9`.


In [ ]:
# BEGIN SOLUTION
respuesta_mcq9 <- "B"
# END SOLUTION

In [ ]:
. = ottr::check("tests/p2_mcq9_contingencia_interpretacion.R")

---
## 6. Gasto promedio según nivel de membresía

El área financiera evalúa si vale la pena invertir más en atraer y retener clientes de membresía Oro, o si el retorno de ese segmento no compensa el costo de sus beneficios. Para decidir, necesitan una cifra concreta: cuánto gasta en promedio un cliente de cada nivel.

**Función recomendada:** `tapply()` o `aggregate()` — [documentación de tapply](https://www.rdocumentation.org/packages/base/versions/3.6.2/topics/tapply).


**Pregunta 6.1 (código):** Calcula el gasto mensual promedio por nivel de membresía en una variable `gasto_por_membresia`. Guarda el promedio del grupo **Oro** en `gasto_promedio_oro` y el del grupo **Bronce** en `gasto_promedio_bronce`.

In [ ]:
# BEGIN SOLUTION
gasto_por_membresia <- tapply(muestra$gasto_mensual, muestra$tipo_membresia, mean)
gasto_promedio_oro <- as.numeric(gasto_por_membresia["Oro"])
gasto_promedio_bronce <- as.numeric(gasto_por_membresia["Bronce"])
# END SOLUTION
gasto_por_membresia

In [ ]:
. = ottr::check("tests/p2_agrupado.R")

**Pregunta 6.2 (opción múltiple):** ¿qué patrón describe mejor la relación entre membresía y gasto?

- A) Bronce gasta más en promedio que Oro.
- B) Oro y Bronce gastan prácticamente lo mismo en promedio.
- C) Oro gasta notablemente más en promedio que Bronce.
- D) No hay forma de comparar el gasto promedio entre grupos.

Guarda la letra en `respuesta_mcq10`.


In [ ]:
# BEGIN SOLUTION
respuesta_mcq10 <- {g <- tapply(muestra$gasto_mensual, muestra$tipo_membresia, mean); r <- g[["Oro"]] / g[["Bronce"]]; if (r > 1.1) "C" else if (r < 1/1.1) "A" else "B"}  # en la mayoría de las muestras: "C"
# END SOLUTION
respuesta_mcq10

In [ ]:
. = ottr::check("tests/p2_mcq10_gasto_membresia.R")

---
## 7. Perfil etario de la base de clientes

Antes de diseñar el mensaje de una nueva campaña, el equipo de marketing necesita saber si debe dirigirse a un público joven, a uno mayor, o si la base de clientes agrupa edades muy diversas — la forma de la distribución de la edad orienta esa decisión.

**Función recomendada:** `hist()` — [documentación](https://www.rdocumentation.org/packages/graphics/versions/3.6.2/topics/hist).


In [ ]:
# Genera un histograma de la edad de tu muestra (ajusta breaks si lo consideras necesario)
hist(muestra$edad,
     main = "Distribucion de la edad de los clientes",
     xlab = "Edad (anios)",
     col = "grey85",
     border = "white")


**Pregunta 7.1 (opción múltiple):** con base en el histograma que acabas de generar, ¿cómo describirías la forma de la distribución de la edad?

- A) Fuertemente asimétrica a la derecha, con muchos clientes muy jóvenes.
- B) Aproximadamente simétrica, similar a una campana.
- C) Bimodal, con dos grupos de edad claramente separados.
- D) Uniforme: todas las edades son igual de frecuentes.

Guarda la letra en `respuesta_mcq11`.


In [ ]:
# BEGIN SOLUTION
respuesta_mcq11 <- "B"
# END SOLUTION

In [ ]:
. = ottr::check("tests/p2_mcq11_histograma.R")

---
## 8. Comparación visual del gasto entre membresías

Complementando el análisis numérico de la sección 6, el equipo quiere confirmar visualmente si el patrón de gasto por membresía es consistente, o si hay clientes atípicos dentro de cada grupo que podrían estar sesgando esa conclusión.

**Función recomendada:** `boxplot()`, con una fórmula tipo `variable_numerica ~ variable_categorica` — [documentación](https://www.rdocumentation.org/packages/graphics/versions/3.6.2/topics/boxplot).


In [ ]:
# Genera un diagrama de cajas y bigotes del gasto mensual, separado por tipo de membresia
boxplot(gasto_mensual ~ tipo_membresia, data = muestra,
        main = "Gasto mensual por tipo de membresia",
        xlab = "Tipo de membresia",
        ylab = "Gasto mensual",
        col = "grey85")


**Pregunta 8.1 (opción múltiple):** ¿qué observas al comparar las cajas de los tres grupos?

- A) Las medianas de gasto son prácticamente idénticas entre los tres grupos.
- B) Oro tiene la mediana de gasto más alta de los tres grupos, y probablemente veas puntos atípicos por encima de las cajas.
- C) Bronce tiene la mediana de gasto más alta de los tres grupos.
- D) Plata tiene la mediana de gasto más alta de los tres grupos.

Guarda la letra en `respuesta_mcq12`.


In [ ]:
# BEGIN SOLUTION
respuesta_mcq12 <- {m <- tapply(muestra$gasto_mensual, muestra$tipo_membresia, median); c(Oro = "B", Bronce = "C", Plata = "D")[[names(which.max(m))]]}  # en la mayoría de las muestras: "B"
# END SOLUTION
respuesta_mcq12

In [ ]:
. = ottr::check("tests/p2_mcq12_boxplot.R")

---
## 9. Relación entre frecuencia de compra y gasto

El equipo de marketing está diseñando un programa que incentive visitas más frecuentes a la tienda, bajo la hipótesis de que comprar con más frecuencia se traduce en mayor gasto total. Antes de invertir en ese programa, quieren confirmar si esa relación realmente se observa en los datos.

**Función recomendada:** `plot()` — [documentación](https://www.rdocumentation.org/packages/graphics/versions/3.6.2/topics/plot.default).


In [ ]:
# Genera un diagrama de dispersion: frecuencia de compra (eje x) vs gasto mensual (eje y)
plot(muestra$frecuencia_compra_mensual, muestra$gasto_mensual,
     main = "Frecuencia de compra vs. gasto mensual",
     xlab = "Frecuencia de compra mensual",
     ylab = "Gasto mensual",
     pch = 19, col = rgb(0, 0, 0, 0.4))


**Pregunta 9.1 (opción múltiple):** ¿qué tipo de relación observas entre la frecuencia de compra y el gasto mensual?

- A) Relación positiva fuerte: a mayor frecuencia de compra, claramente mayor gasto mensual.
- B) Relación negativa fuerte: a mayor frecuencia de compra, claramente menor gasto mensual.
- C) Relación débil o casi nula: los puntos están muy dispersos y comprar con más frecuencia no garantiza un mayor gasto.
- D) Relación perfectamente lineal, sin dispersión.

Guarda la letra en `respuesta_mcq13`.


In [ ]:
# BEGIN SOLUTION
respuesta_mcq13 <- "C"
# END SOLUTION

In [ ]:
. = ottr::check("tests/p2_mcq13_dispersion.R")

---
## Cierre — el perfil de tu muestra

<!-- BEGIN QUESTION -->

**Pregunta final (reflexión, calificada manualmente):** con base en todos los cálculos y gráficos anteriores, escribe **un párrafo** describiendo el perfil de los clientes en tu muestra: ¿quiénes son (región, género, edad típica), qué tan leales son (antigüedad, frecuencia), y cómo se comportan en gasto (nivel, dispersión, relación con la membresía)?

_Type your answer here, replacing this text._

**Respuesta modelo:** Un buen párrafo integra: (1) **quiénes son**: región más frecuente y su peso (entre 30 % y 45 %), reparto por género y edad típica (alrededor de 40 años, distribución aproximadamente simétrica); (2) **lealtad**: antigüedad mediana baja (cerca de dos años) con una cola de clientes muy antiguos que la regla de Tukey señala como atípicos, y frecuencia de compra de unas 4 visitas al mes; (3) **gasto**: muy disperso (CV mucho mayor que el de la edad), asimétrico a la derecha y leptocúrtico, por lo que la mediana representa mejor al cliente típico que la media; el gasto crece con el nivel de membresía (Oro gasta en promedio casi el doble que Bronce) y la relación entre frecuencia de compra y gasto es débil. Debe cerrar con una implicación de negocio (por ejemplo, segmentar precios o campañas por membresía en lugar de usar una política única).

<!-- END QUESTION -->

---
## Entrega

1. Verifica que corriste todas las celdas en orden y que no hay errores.
2. Descarga el notebook: `Archivo > Descargar > Descargar .ipynb`.
3. Sube el archivo a Moodle con el nombre `E1-02_Perfilamiento_<tu_codigo>.ipynb`.